<H1>Extract, combine and format the trial data<H1>

## Balaban et al., 2019

The full dataset can be found and downloaded at: https://osf.io/mzs9e/

1. Import the needed packages.

In [1]:
import os
import fnmatch
import pandas as pd
import numpy as np

2. Set the directory and the pattern within the file names to be matched.
3. Open the files and compile them into a list called arrays and a concatenated table called final_array.

Choose one of the options below (create the file list here in Python or use the file list created in R). 

In [2]:
directory = '/Path/to/main_directory/K'
pattern = '*_ChangeDetection_BIGT.txt'

In [3]:
#OPTION 1!! 

#Create a list with each element corresponding to a unique text file
#This works, but it compiles the files in a different order compared with the corresponding R script. 
#filelist = []

#for root, dirs, files in os.walk(directory):
    #for filename in fnmatch.filter(files, pattern):
        #datafile = os.path.join(root, filename)
        #filelist.append(datafile)

#Concatenate the arrays
#arrays = [np.loadtxt(file, dtype='str') for file in filelist]
#final_array = np.concatenate(arrays, axis=0)

In [4]:
#Option 2!!

#Alternatively, Make the dataset using the file list that was created in R to check against the R dataset
with open("/Path/to/R_File_List.txt") as file:
    lines = [line.rstrip() for line in file]
    

#Add the paths and save as filelist_R
lines_final = [line.replace("./", "/Path/to/main_directory/K/") for line in lines]

filelist_R = lines_final
print(filelist_R)

#Concatenate the arrays
arrays = [np.loadtxt(file, dtype='str') for file in filelist_R]
final_array = np.concatenate(arrays, axis=0)

['/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/1_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/13_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/2_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/20_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/22_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/23_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/26_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/28_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/3_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/34_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/K1/K11/K111/36_ChangeDetection_BIGT.txt', '/Users/saramosteller/Documents/Check_Code/K/

In [5]:
len(arrays) #Should be 3849

3849

In [6]:
len(final_array) #Should be 462186

462186

4. Create the final dataframe with an index that is unique to each dataset. 

In [7]:
#Convert the data to a preliminary pandas dataframe

colnames = ("PartID", "set_size", "stim_duration", "change_orig", "response_orig", "rt", "test_location_x", "test_location_y")
data_main = pd.DataFrame(final_array, columns = colnames)
data_main.head(5)

,PartID,set_size,stim_duration,change_orig,response_orig,rt,test_location_x,test_location_y
0,1,4,150,0,102,749,300,300
1,1,8,150,5,101,1142,-300,-300
2,1,4,150,5,101,1644,300,-200
3,1,8,150,5,2,1192,300,-300
4,1,4,150,0,1,938,-200,-100


In [8]:
#Check the data types
data_main.dtypes

PartID             object
set_size           object
stim_duration      object
change_orig        object
response_orig      object
rt                 object
test_location_x    object
test_location_y    object
dtype: object

In [9]:
#Check the dimensions
data_main.shape

(462186, 8)

In [10]:
#Every column is a string, so try converting to integers 

#data_main['PartID'] = data_main['PartID'].astype('int') #This is where the string error is-the other columns convert to integers
data_main['set_size'] = data_main['set_size'].astype('int')
data_main['stim_duration'] = data_main['stim_duration'].astype('int')
data_main['change_orig'] = data_main['change_orig'].astype('int')
data_main['response_orig'] = data_main['response_orig'].astype('int')
data_main['rt'] = data_main['rt'].astype('int')
data_main['test_location_x'] = data_main['test_location_x'].astype('int')
data_main['test_location_y'] = data_main['test_location_y'].astype('int')
data_main.dtypes

PartID             object
set_size            int64
stim_duration       int64
change_orig         int64
response_orig       int64
rt                  int64
test_location_x     int64
test_location_y     int64
dtype: object

In [11]:
#Get rid of the PartID column

data_main = data_main.drop(['PartID'], axis = 1)
data_main.head()

,set_size,stim_duration,change_orig,response_orig,rt,test_location_x,test_location_y
0,4,150,0,102,749,300,300
1,8,150,5,101,1142,-300,-300
2,4,150,5,101,1644,300,-200
3,8,150,5,2,1192,300,-300
4,4,150,0,1,938,-200,-100


In [12]:
#Create a dataset index column
len(arrays[6][:,1:7]) #Check one dataset per array index

120

In [13]:
type(arrays)

list

In [14]:
#Outputs an ID column for the final dataframe

def create_index_column(inputlist) :
    # Create the index column
    
    length = [len(array) for array in inputlist]
    index = pd.DataFrame(length).reset_index()
    index.rename(columns={0: 'length'}, inplace=True)
    index['id'] = index.index
    indexcol = index.iloc[index.index.repeat(index['length'])]
    indexcol['trial_num'] = indexcol.index
    indexcol['trial_num'] = indexcol.groupby('index').rank(method='first').astype(int)
    ID = pd.DataFrame(indexcol[['id','trial_num']]).reset_index(drop=True)
       
    return ID

In [15]:
#Iterate through the list and get the length of each array
lengthlist = []
for array in arrays :
    length = len(array)
    lengthlist.append(length)
    cols = ['length']
    lengths = pd.DataFrame(lengthlist, columns=cols)


In [16]:
#Create the unique participant ID and row number information in separate data frame
index = pd.DataFrame(lengths).reset_index()
index['id'] = index.index
indexcol = index.iloc[index.index.repeat(index['length'])]
indexcol = pd.DataFrame(indexcol).reset_index()
count_by_id = pd.DataFrame(indexcol[['id', 'length']]).reset_index(drop=True)
count_by_id['trial_num'] = count_by_id.groupby('id').cumcount()
count_by_id = count_by_id[['id', 'trial_num']]

In [17]:
#Concatenate these to create the final dataframe
data = pd.concat([count_by_id, data_main], axis=1).reset_index(drop=True)
len(data)

462186

In [18]:
#Number variables starting at 1
data['id'] = data['id']+1
data['trial_num'] = data['trial_num']+1

In [19]:
#Check the final dataset
data.head()

,id,trial_num,set_size,stim_duration,change_orig,response_orig,rt,test_location_x,test_location_y
0,1,1,4,150,0,102,749,300,300
1,1,2,8,150,5,101,1142,-300,-300
2,1,3,4,150,5,101,1644,300,-200
3,1,4,8,150,5,2,1192,300,-300
4,1,5,4,150,0,1,938,-200,-100


In [20]:
#Save as a .csv file
data.to_csv('raw_trial_data.csv')

5. Find the datasets that have > 120 trials and see if the number of trial types is equal across conditions if the extra trials are omitted.

In [21]:
#Count up the trials in each type (set size x same/diff) for each dataset
trialtype_counts = data.groupby(['id','set_size','change_orig']).count().reset_index()
trialtype_counts

#Filter out the datasets with more than 30 trials per trial type
anom_filter = trialtype_counts['stim_duration'] != 30
filtered_counts = trialtype_counts[anom_filter]

#Write these participants to a csv file to inspect
filtered_counts.to_csv('filtered_counts.csv')

#Write their raw data to a csv file to inspect
filter_IDs = filtered_counts['id'].unique() #There are 54 participants with more than 120 trials
filtered_data = data[data['id'].isin(filter_IDs)]
filtered_data.to_csv('filtered_data.csv')

In [22]:
#Alternatively
#Count up the trials in each type (set size x same/diff) for each dataset 
#after omitting any greater than 120 trials

#Create the trimmed dataset
data['trial_num'] = data.groupby(['id']).cumcount()
trialnum_filter = data['trial_num'] <= 120
data_analysis = data[trialnum_filter]
len(data_analysis)
data_analysis.to_csv('trimmed_dataset.csv')

#Count up the trials in each type (set size x same/diff) for each dataset
trialtype_counts = data_analysis.groupby(['id','set_size','change_orig']).count().reset_index()
trialtype_counts

#Filter out the datasets with more than 30 trials per trial type
anom_filter = trialtype_counts['stim_duration'] != 30
filtered_counts = trialtype_counts[anom_filter]
filtered_counts
filtered_counts.to_csv('filtered_counts_after_removing_trials_over_120.csv')

#This doesn't result in an even number of trial types per condition for these participants,
#So, use the original dataset

6. Format the dataset.

In [23]:
#data = data.drop(data.columns[[0]], axis = 1)
data.head()

,id,trial_num,set_size,stim_duration,change_orig,response_orig,rt,test_location_x,test_location_y
0,1,0,4,150,0,102,749,300,300
1,1,1,8,150,5,101,1142,-300,-300
2,1,2,4,150,5,101,1644,300,-200
3,1,3,8,150,5,2,1192,300,-300
4,1,4,4,150,0,1,938,-200,-100


In [24]:
data['response'] = data['response_orig'].map({1:0, 2:1, 101:0, 102:1})
data['change'] = data['change_orig'].map({0:0, 5:1})
data.dtypes

id                   int64
trial_num            int64
set_size             int64
stim_duration        int64
change_orig          int64
response_orig        int64
rt                   int64
test_location_x      int64
test_location_y      int64
response           float64
change               int64
dtype: object

In [25]:
data['change'].isna().sum()

np.int64(0)

In [26]:
data['response'].isna().sum()

np.int64(37)

In [27]:
data['response_orig'].isna().sum() #The original response variable doesn't have any missing values, so these need to be filled in in the choice variable

np.int64(0)

In [28]:
missing = data[data['response'].isna()]
missing

,id,trial_num,set_size,stim_duration,change_orig,response_orig,rt,test_location_x,test_location_y,response,change
94717,790,10,8,150,0,103,7433,200,200,NaN,0
124722,1040,0,8,150,5,103,10915,-300,200,NaN,1
136369,1137,1,8,150,0,103,1018,300,200,NaN,0
237409,1977,111,4,150,5,103,282,-200,-100,NaN,1
237410,1977,112,4,150,5,103,502,100,200,NaN,1
264452,2203,29,8,150,5,103,12343,100,200,NaN,1
274385,2286,2,4,150,0,103,3222,100,100,NaN,0
274389,2286,6,8,150,0,103,4361,100,-100,NaN,0
274396,2286,13,8,150,5,103,1780,-200,300,NaN,1
274397,2286,14,8,150,0,103,2377,100,100,NaN,0


In [29]:
data['rt'].isna().sum()

np.int64(0)

In [30]:
#There are 37 NA obvs in the response variable, so one option is to get rid of these rows.
#data = data[data['response'].notna()]
#data['response'] = data['response'].astype(int)
#data.shape

In [31]:
#Alternatively, delete these datasets from the dataframe. This is what was used in this study. 
odd_response_ids = data[['id']][data['response'].isna()]
data = data[~data['id'].isin(odd_response_ids['id'])]
data['response'] = data['response'].astype('int')
data.shape

(460866, 11)

In [32]:
#Select and order the columns to keep
data_final = data[['id', 'trial_num', 'set_size', 'change', 'response', 'rt', 'stim_duration', 'test_location_x', 'test_location_y']]
data_final.head()

,id,trial_num,set_size,change,response,rt,stim_duration,test_location_x,test_location_y
0,1,0,4,0,1,749,150,300,300
1,1,1,8,1,0,1142,150,-300,-300
2,1,2,4,1,0,1644,150,300,-200
3,1,3,8,1,1,1192,150,300,-300
4,1,4,4,0,0,938,150,-200,-100


In [33]:
#Write the processed trial-level dataset to a new csv file
data_final.to_csv('Trial_Data.csv')